# Chapter 6 — WaveNet: depth, hierarchy, and a real development workflow

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 6 — WaveNet: depth, hierarchy, and a real development workflow

**Video:** 56m · [youtu.be/t3YJ5hKiMQ0](https://youtu.be/t3YJ5hKiMQ0) · **Based on:** WaveNet (DeepMind, 2016), originally a raw-audio generator. [transcript]

### The problem

The Chapter 3 network takes 3 characters, mashes all their embeddings together in a single step, and squeezes everything through one hidden layer. Two flaws: the context is short, and cramming it all into one layer wastes the network's depth.

### The idea: combine information in a tree, not a heap

Rather than merging 8 characters at once, merge them in pairs, then merge the pairs, then merge those. Eight becomes four, then two, then one. Each layer performs one modest fusion instead of one enormous one.

**Analogy.** A single-elimination tournament versus a 64-player free-for-all. The tournament plays the same number of games, but every match compares two comparable things, and information moves upward in stages.

> **Say it to a six-year-old.** If eight friends all shout their favourite colour at you at once, you hear nothing. But if they pair up and each pair agrees on one colour, then those four pairs pair up again, and so on, by the end you get one answer and you heard every single person along the way.

### Step 1 — lengthen the context and rebuild as modules

Context goes from 3 characters to **8** [transcript], which gives 182,625 training examples of "eight characters predict the ninth."

The code gets rewritten as classes matching PyTorch's own `torch.nn` API. This is partly organization and partly demystification: after writing them, PyTorch's module system holds no surprises.

**Run it.**

In [ ]:
import torch
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5   # Kaiming, Chapter 4
        self.bias = torch.zeros(fan_out) if bias else None
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out = self.out + self.bias
        return self.out
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self):
        return []

class Sequential:
    def __init__(self, layers): self.layers = layers
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

Every one of these has a direct counterpart in `torch.nn`, with the same name and nearly the same code.

### Step 2 — the layer that does the tree

**Run it.** This is the one genuinely new piece:

In [ ]:
class FlattenConsecutive:
    def __init__(self, n): self.n = n     # how many neighbours to fuse
    def __call__(self, x):
        B, T, C = x.shape                 # batch, time (characters), channels
        x = x.view(B, T//self.n, C*self.n)   # fuse n neighbours into one wider vector
        if x.shape[1] == 1:
            x = x.squeeze(1)              # drop the time axis once it is length 1
        self.out = x
        return self.out
    def parameters(self): return []

# watch the shapes collapse in a tree
e = torch.randn(4, 8, 10)          # 4 examples, 8 characters, 10 numbers each
print("start:            ", tuple(e.shape))
a = FlattenConsecutive(2)(e); print("after 1st fuse:   ", tuple(a.shape))
b = FlattenConsecutive(2)(a); print("after 2nd fuse:   ", tuple(b.shape))
c = FlattenConsecutive(2)(b); print("after 3rd fuse:   ", tuple(c.shape))

**What you should see:**

**Expected output:**

```
start:             (4, 8, 10)
after 1st fuse:    (4, 4, 20)
after 2nd fuse:    (4, 2, 40)
after 3rd fuse:    (4, 80)
```

[verified]

Read the middle number: 8 characters, then 4 groups, then 2, then 1. Read the last: 10 numbers per position, then 20, then 40, then 80. **Information is being merged in stages instead of all at once.** In the real network a `Linear` and a `Tanh` sit between each fuse, so each merge is followed by actual computation.

Compare with Chapter 3, which went straight from `(4, 8, 10)` to `(4, 80)` in one step [transcript]. Same numbers at the end, completely different path.

**Run it.** The whole network assembled from the classes above, with every intermediate shape printed:

In [ ]:
n_embd, n_hidden = 10, 68
C = torch.randn(27, n_embd)
model = Sequential([
    FlattenConsecutive(2), Linear(n_embd*2,  n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden*2, n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden*2, n_hidden), Tanh(),
    Linear(n_hidden, 27),
])
x = torch.randint(0, 27, (4, 8))       # 4 examples, 8 characters each
out = model(C[x])
print("output:", tuple(out.shape))
print("parameters:", sum(p.numel() for p in model.parameters()) + C.numel())
for layer in model.layers:
    print(f"  {layer.__class__.__name__:20} {tuple(layer.out.shape)}")

**What you should see:**

**Expected output:**

```
output: (4, 27)
parameters: 22193
  FlattenConsecutive   (4, 4, 20)
  Linear               (4, 4, 68)
  Tanh                 (4, 4, 68)
  FlattenConsecutive   (4, 2, 136)
  Linear               (4, 2, 68)
  Tanh                 (4, 2, 68)
  FlattenConsecutive   (4, 136)
  Linear               (4, 68)
  Tanh                 (4, 68)
  Linear               (4, 27)
```

[verified]

**22,193 parameters**, matching the lecture's "about 22,000" [transcript]. Read the shape column downward and you can watch the tree: 8 positions become 4, then 2, then 1, while the channel count doubles at each fuse and is immediately projected back down to 68 by the following `Linear`. That expand-then-project rhythm is the whole architecture.

### Step 3 — the bug that justifies the lecture

Adding a dimension to the tensors silently broke BatchNorm. It computed means and variances over the wrong axes, maintaining statistics for the wrong grouping of channels [transcript]. Nothing crashed. The loss curve looked plausible.

**Run it.** See the bug directly:

In [ ]:
x = torch.randn(32, 4, 68)                 # batch, time, channels
print("mean over dim 0 only:  ", tuple(x.mean(0, keepdim=True).shape), "<- 4x68 = 272 statistics, WRONG")
print("mean over dims 0 and 1:", tuple(x.mean((0,1), keepdim=True).shape), "<- 68 statistics, correct")

**What you should see:**

**Expected output:**

```
mean over dim 0 only:   (1, 4, 68) <- 4x68 = 272 statistics, WRONG
mean over dims 0 and 1: (1, 1, 68) <- 68 statistics, correct
```

[verified]

Both run. Both produce a usable tensor. Only one is the operation you intended. Karpathy finds it by printing shapes and inspecting the running-statistics buffer.

**The lesson, which is the real subject of this chapter: most deep learning bugs are shape bugs, and they do not raise exceptions.** They produce a model that trains to a worse number than it should, and without a baseline you would never know.

### Step 4 — results

| Model | Parameters | Validation loss |
|---|---|---|
| Bigram (Chapter 2) | 729 | 2.4544 [verified] |
| MLP, 3 characters (Chapter 3) | 11,897 | 2.2778 [verified] |
| Fixed init (Chapter 4) | 11,897 | 2.1481 [verified] |
| WaveNet-style, 8 characters | ~22,000 | 2.029 [transcript] |
| Same, widened | 76,000 | **1.993** [transcript] |

Crossing below 2.0 for the first time. Karpathy also flags the cost: "the training takes a lot longer" and "we are starting to have to wait" [transcript]. That waiting is what Chapter 9 attacks.

### Step 5 — the workflow, stated explicitly

Karpathy narrates the actual working pattern of a practitioner, which almost no course shows:

- Keep a Jupyter notebook for experiments and the PyTorch documentation open in a browser tab beside it.
- Read the layer documentation while writing the layer. Remembering argument order is not a skill.
- Print tensor shapes constantly, and check them against what you expected before running anything long.
- Change one thing per experiment and record the resulting validation loss in a list. That list is the actual work product.

### What the chapter skips, and says so

Real WaveNet uses gated activations and residual and skip connections, not this plain stack. Karpathy leaves them out to keep the structure legible; the transformer in Chapter 7 introduces residual connections properly.

> **For the PhD in the room.** The tree here is a dilated causal convolution with dilation doubling per layer, so the receptive field grows as 2^depth while parameters grow linearly, which is exactly WaveNet's contribution over a plain causal conv stack. Worth noting what this architecture cannot do that Chapter 7's can: the fusion pattern is fixed and content-independent, so position 3 always merges with position 4 regardless of what they contain. Attention replaces that fixed tree with a learned, input-dependent, all-pairs gather, at a cost of O(T²) rather than O(T log T). The recent state-space literature (S4, Mamba) is largely an attempt to recover subquadratic scaling while keeping content dependence, so this chapter's structure is closer to the current research frontier than its 2016 date suggests.

### Exercises

1. **Print shapes at every layer** of the full network with a loop over `model.layers`, checking each against what you predicted.
2. **Change the fuse width** from 2 to 4 with 8 characters of context. That gives a shallower tree. Compare the validation loss.
3. **Reproduce the BatchNorm bug** by normalizing over dimension 0 only in a 3-D network, train it, and measure how much loss it silently costs.
4. **Compare fairly.** Train a Chapter 3 flat MLP with the same parameter count as the WaveNet version. Some of the improvement is the tree; some is just more parameters and more context. Find out how much of each.

### 30-second version

Instead of dumping eight characters into one layer at once, fuse them two at a time in a tree, so each layer does a modest, meaningful piece of work, and the loss drops below 2.0 for the first time. The chapter doubles as an honest demonstration of the job: a shape bug that breaks the model without breaking the code, found by printing tensor shapes and reading the documentation.

---